In [ ]:
/**
 * PLAYBOARD 조회수 순위 크롤러
 *
 * 사용법:
 * 1. playboard.co/chart/most-viewed (한국) 페이지로 이동
 * 2. F12 → Console 탭 클릭
 * 3. 이 코드 전체를 붙여넣기 후 Enter
 * 4. 완료되면 CSV 파일 자동 다운로드
 */

(async function playboardScraper() {

  // ── 설정 ──────────────────────────────────────────────
  const TARGET_COUNT = 200;       // 수집할 순위 수
  const SCROLL_WAIT_MS = 1500;    // 스크롤 후 대기 시간 (ms)
  const MAX_SCROLL_ATTEMPTS = 15; // 최대 스크롤 시도 횟수
  // ──────────────────────────────────────────────────────

  console.log('🚀 플레이보드 크롤러 시작...');

  // 1. 스크롤해서 모든 항목 로드
  async function loadAllItems() {
    let prevCount = 0;
    let stuckCount = 0;

    for (let i = 0; i < MAX_SCROLL_ATTEMPTS; i++) {
      window.scrollTo(0, document.body.scrollHeight);
      await new Promise(r => setTimeout(r, SCROLL_WAIT_MS));

      const currentCount = document.querySelectorAll('tr.chart__row:not(.chart__row--ad)').length;
      console.log(`  로딩 중... ${currentCount}개 항목 발견`);

      if (currentCount >= TARGET_COUNT) break;

      if (currentCount === prevCount) {
        stuckCount++;
        if (stuckCount >= 3) {
          console.log('  더 이상 새 항목 없음. 로딩 완료.');
          break;
        }
      } else {
        stuckCount = 0;
      }

      prevCount = currentCount;
    }

    window.scrollTo(0, 0); // 상단으로 복귀
    await new Promise(r => setTimeout(r, 500));
  }

  await loadAllItems();

  // 2. 데이터 추출
  const rows = document.querySelectorAll('tr.chart__row:not(.chart__row--ad)');
  console.log(`\n📊 총 ${rows.length}개 행에서 데이터 추출 중...`);

  const results = [];

  rows.forEach((row, idx) => {

    // ── 순위 ──
    const rankTd = row.querySelector('td.rank');
    const rankNum = rankTd?.querySelector('.rank__number, [class*="number"]')?.textContent?.trim()
      || (idx + 1).toString();

    // ── 순위 변동 ──
    // 올랐으면 "+숫자" 내렸으면 "-숫자" 신규/동일이면 "NEW" 또는 "-"
    const changeEl = rankTd?.querySelector('[class*="change"], [class*="diff"], .rank__change');
    let rankChange = changeEl?.textContent?.trim() || '-';
    // 화살표 아이콘 제거
    rankChange = rankChange.replace(/[▲▼↑↓]/g, '').trim();
    if (!rankChange) rankChange = '-';

    // ── 제목 영역 ──
    const titleTd = row.querySelector('td.title');

    // 비디오 링크 + 제목
    const titleAnchor = titleTd?.querySelector('a.title__label, a[href*="/video/"]');
    const videoTitle = titleAnchor?.getAttribute('title')
      || titleAnchor?.querySelector('[class*="label"]')?.textContent?.trim()
      || titleAnchor?.textContent?.trim()
      || '';

    // 비디오 리포트 링크
    const videoHref = titleAnchor?.getAttribute('href') || '';
    const videoReportLink = videoHref ? `https://playboard.co${videoHref}` : '';

    // 유튜브 영상 링크 (href에서 video ID 추출)
    const videoIdMatch = videoHref.match(/\/video\/([^?/]+)/);
    const videoId = videoIdMatch?.[1] || '';
    const youtubeVideoLink = videoId ? `https://www.youtube.com/watch?v=${videoId}` : '';

    // ── 카테고리 ──
    const tagEls = titleTd?.querySelectorAll('.ttags a, [class*="ttag"] a, [class*="tag"] a') || [];
    const category = Array.from(tagEls).map(t => t.textContent.trim()).join(' / ') || '-';

    // ── 게시 날짜 ──
    const dateEl = titleTd?.querySelector('.title__date, [class*="date"]');
    const publishDate = dateEl?.textContent?.trim() || '';

    // ── 조회수 ──
    const scoreTd = row.querySelector('td.score, td[class*="score"]');
    // 숫자만 추출 (쉼표 포함)
    const viewCountRaw = scoreTd?.textContent?.trim() || '';
    const viewCount = viewCountRaw.replace(/\s+/g, '') || '-';

    // ── 채널 ──
    const channelTd = row.querySelector('td.channel, td[class*="channel"]');
    const channelAnchor = channelTd?.querySelector('a[href*="/channel/"]');
    const channelName = channelAnchor?.textContent?.trim() || '';

    const channelHref = channelAnchor?.getAttribute('href') || '';
    const channelReportLink = channelHref ? `https://playboard.co${channelHref}` : '';

    // 채널 유튜브 링크 (href에서 channel ID 추출)
    // playboard 채널 URL 형식: /channel/CHANNEL_ID 또는 /channel/CHANNEL_ID/video
    const channelIdMatch = channelHref.match(/\/channel\/([^/?]+)/);
    const channelId = channelIdMatch?.[1] || '';
    const youtubeChannelLink = channelId ? `https://www.youtube.com/channel/${channelId}` : '';

    results.push({
      '순위': rankNum,
      '순위변동': rankChange,
      '영상제목': videoTitle,
      '조회수': viewCount,
      '카테고리': category,
      '게시날짜': publishDate,
      '채널명': channelName,
      '비디오_리포트링크': videoReportLink,
      '유튜브_영상링크': youtubeVideoLink,
      '채널_리포트링크': channelReportLink,
      '유튜브_채널링크': youtubeChannelLink,
    });
  });

  // 3. CSV 변환
  console.log('\n📁 CSV 파일 생성 중...');

  const headers = Object.keys(results[0]);
  const escape = (val) => `"${String(val ?? '').replace(/"/g, '""')}"`;

  const csvLines = [
    headers.map(escape).join(','),
    ...results.map(row => headers.map(h => escape(row[h])).join(','))
  ];

  // BOM 포함 (한글 엑셀 호환)
  const csvContent = '﻿' + csvLines.join('\r\n');
  const blob = new Blob([csvContent], { type: 'text/csv;charset=utf-8;' });
  const url = URL.createObjectURL(blob);

  const today = new Date().toISOString().slice(0, 10).replace(/-/g, '');
  const filename = `playboard_조회수순위_${today}.csv`;

  const a = document.createElement('a');
  a.href = url;
  a.download = filename;
  document.body.appendChild(a);
  a.click();
  document.body.removeChild(a);
  URL.revokeObjectURL(url);

  // 4. 결과 요약
  console.log(`\n✅ 완료! ${results.length}개 데이터를 "${filename}"으로 저장했습니다.`);
  console.log('\n📋 상위 3개 미리보기:');
  console.table(results.slice(0, 3));

  return results;

})();
